# AACBR Interpretability

Build a case base with one **fixed configuration** (no sweeping) and explain predictions for new test cases via:
- **Argumentation subgraph** — one panel per ordinal threshold: relevant precedents, neutralised cases, grounded-extension result
- **Textual summary** — which cases argued for/against the prediction

The **BraTS** and **BrainWear** sections are independent. Run either or both.

In [2]:
import sys, csv as csv_mod, re
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import networkx as nx
from tqdm import tqdm
from sklearn.model_selection import train_test_split

FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

from aacbr.aacbr_parallel import AACBRParallel
from aacbr.configs.brats_model_config import BraTSOutcomeConfig
from utils.characterisations import (
    TumourCharacterisationLarge2D, TumourCharacterisationSmall2D,
    TumourCharacterisationSmall2DV2, TumourCharacterisationLarge2DV2,
)

CHAR_MODELS = {
    'small':    TumourCharacterisationSmall2D,
    'large':    TumourCharacterisationLarge2D,
    'small_v2': TumourCharacterisationSmall2DV2,
    'large_v2': TumourCharacterisationLarge2DV2,
}
CONFIG = str(FYP_ROOT / 'aacbr' / 'configs' / 'brats_configs' / 'flat_config.json')

## Shared interpretability helpers
*(run once before either dataset section)*

In [43]:
# ── Utilities ───────────────────────────────────────────────────────────

def _dedup(feats, out):
    unique, inverse = np.unique(feats, axis=0, return_inverse=True)
    dedup_out = np.array([int(np.round(np.median(out[inverse == i])))
                          for i in range(len(unique))])
    return unique, dedup_out


def _fit_ordinal_models(train_feats, train_out, char_model, cfg, n_bins,
                        strict=True, use_supports=False, dedup=True,
                        strategy='ordinal'):
    ls_fn = lambda a, b: char_model.less_specific(a, b, strict=strict)
    f, o = _dedup(train_feats, train_out) if dedup else (train_feats, train_out)
    models = {}
    for k in range(n_bins):
        if strategy == 'flat':
            labels = (o == k).astype(int)
        else:  # ordinal
            labels = (o >= k).astype(int)
        m = AACBRParallel(
            less_specific=ls_fn,
            default_case=char_model.default_case(),
            default_outcome=cfg.default_outcome,
            include_supports=use_supports,
            supported_attack_chain=use_supports,
        )
        m.fit(f, labels)
        models[k] = m
    return models


def describe_features(feat_vec, char_model):
    """Return list of (feature_name, count) for every non-zero feature."""
    names = char_model.feature_names()
    return [(names[i], int(feat_vec[i])) for i in np.flatnonzero(feat_vec)]


def _abbrev(feat_vec, char_model, n=4):
    pairs = describe_features(feat_vec, char_model)
    parts = [f'{name}x{cnt}' for name, cnt in pairs[:n]]
    if len(pairs) > n:
        parts.append(f'+{len(pairs)-n} more')
    return ', '.join(parts) if parts else 'empty'


# ── Argumentation subgraph (one binary threshold model) ─────────────────

def plot_argumentation_subgraph(model_k, new_case_feat, char_model, k,
                                ax=None, title='', show_isolated=False,
                                show_new_case=False, show_defense_arrows=True,
                                exclude_nodes=None, legend_loc='lower left',
                                ranksep=0.9, nodesep=0.4):
    """Draw argumentation subgraph for threshold k.

    get_new_case_attacks_mask semantics:
      mask[i]=True  -> casebase[i] NOT a subset of new case -> neutralised
      mask[i]=False -> casebase[i] IS  a subset of new case -> relevant precedent

    show_isolated:  if False (default), remove degree-0 nodes from the plot.
    show_new_case:  if True, add the New Case node with green arrows to the
                    grounded cases it defends.
    """
    nf = new_case_feat[np.newaxis, :] if new_case_feat.ndim == 1 else new_case_feat
    attacks_mask = model_k.get_new_case_attacks_mask(nf)[0]
    grounded     = model_k.compute_grounded(attacks_mask[np.newaxis, :])[0]

    default_idx     = model_k.default_index
    binary_outcomes = model_k.casebase_outcomes

    relevant_ids = np.where(~attacks_mask)[0]
    new_case_id  = len(model_k.casebase_features)
    _excluded = set(exclude_nodes) if exclude_nodes else set()
    relevant_ids = np.array([i for i in relevant_ids if i not in _excluded])

    am = model_k.attacks_matrix
    if hasattr(am, 'toarray'):
        am = am.toarray()
    edges = [(int(s), int(t))
             for s in relevant_ids for t in relevant_ids
             if s != t and am[s, t]]

    G = nx.DiGraph()
    G.add_nodes_from(list(relevant_ids) + ([new_case_id] if show_new_case else []))
    G.add_edges_from(edges)

    if not show_isolated:
        keep = {new_case_id} if show_new_case else set()
        isolated = [n for n in list(G.nodes())
                    if G.degree(n) == 0 and n not in keep]
        G.remove_nodes_from(isolated)

    G_nodes = list(G.nodes())
    colors, borders, sizes = [], [], []
    for n in G_nodes:
        if n == new_case_id:
            colors.append('#4a90d9')
            borders.append('#1a5fa8')
            sizes.append(900)
        elif n == default_idx:
            colors.append('#6abf69' if grounded[n] else '#e57373')
            borders.append('black')
            sizes.append(1100)
        elif grounded[n]:
            colors.append('#6abf69')
            borders.append('none')
            sizes.append(700)
        else:
            colors.append('#e57373')
            borders.append('none')
            sizes.append(700)

    lbls = {}
    for n in G_nodes:
        if n == new_case_id:
            lbls[n] = 'New\nCase'
        elif n == default_idx:
            lbls[n] = f'Default\nout={int(binary_outcomes[n])}'
        else:
            lbls[n] = f'[{n}]\nout={int(binary_outcomes[n])}'

    # Green edges: new case defends each grounded case in the subgraph
    defense_edges = []
    if show_new_case and show_defense_arrows and new_case_id in G_nodes:
        defense_edges = [(new_case_id, n) for n in G_nodes
                         if n != new_case_id and grounded[n]]

    try:
        pos = nx.nx_agraph.graphviz_layout(
            G, prog='dot', args=f'-Granksep={ranksep} -Gnodesep={nodesep}')
    except Exception:
        pos = nx.spring_layout(G, seed=42)

    # Push nodes adjacent to Default further away so attack arrows are visible
    if default_idx in pos and len(pos) > 1:
        d_x, d_y = pos[default_idx]
        all_vals = list(pos.values())
        scale = max(
            max(p[0] for p in all_vals) - min(p[0] for p in all_vals),
            max(p[1] for p in all_vals) - min(p[1] for p in all_vals),
            1.0,
        )
        min_dist = scale * 0.35
        edges_set = set(edges)
        for n, (nx_, ny_) in list(pos.items()):
            if n == default_idx or n == new_case_id:
                continue
            if (n, default_idx) in edges_set or (default_idx, n) in edges_set:
                dx, dy = nx_ - d_x, ny_ - d_y
                dist = (dx ** 2 + dy ** 2) ** 0.5
                if 0 < dist < min_dist:
                    f = min_dist / dist
                    pos[n] = (d_x + dx * f, d_y + dy * f)

    # If new case is isolated (no edges), place it near the cluster
    if show_new_case and new_case_id in pos and G.degree(new_case_id) == 0:
        other = [v for n, v in pos.items() if n != new_case_id]
        if other:
            cx = sum(x for x, _ in other) / len(other)
            cy = sum(y for _, y in other) / len(other)
            x_range = max(x for x, _ in other) - min(x for x, _ in other)
            y_range = max(y for _, y in other) - min(y for _, y in other)
            offset = max(x_range, y_range) * 0.55
            pos[new_case_id] = (cx + offset, cy + offset)

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))

    ax.set_title(title or f'k={k}', fontsize=10, fontweight='bold')
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes,
                           edgecolors=borders, linewidths=2.5, ax=ax)
    nx.draw_networkx_labels(G, pos, labels=lbls, font_size=6, ax=ax)
    if edges:
        nx.draw_networkx_edges(G, pos, edgelist=edges, ax=ax,
                               arrows=True, arrowstyle='-|>', arrowsize=15,
                               width=1.5, edge_color='#cc3333',
                               connectionstyle='arc3,rad=0.15',
                               min_source_margin=15, min_target_margin=20)
    if defense_edges:
        nx.draw_networkx_edges(G, pos, edgelist=defense_edges, ax=ax,
                               arrows=True, arrowstyle='-|>', arrowsize=15,
                               width=1.5, edge_color='#2ea02e',
                               connectionstyle='arc3,rad=0.15',
                               min_source_margin=15, min_target_margin=20)

    # Legend: only include entries for node/edge types actually present
    legend_handles = []
    if show_new_case and new_case_id in G_nodes:
        legend_handles.append(mpatches.Patch(color='#4a90d9', label='New case'))

    has_grounded = any(n != default_idx and n != new_case_id and grounded[n]
                       for n in G_nodes)
    has_defeated = any(n != default_idx and n != new_case_id and not grounded[n]
                       for n in G_nodes)

    if has_grounded:
        legend_handles.append(mpatches.Patch(color='#6abf69', label='Grounded / defended'))
    if has_defeated:
        legend_handles.append(mpatches.Patch(color='#e57373', label='Defeated'))

    if default_idx in G_nodes:
        do = int(binary_outcomes[default_idx])
        if grounded[default_idx]:
            legend_handles.append(mpatches.Patch(
                facecolor='#6abf69', edgecolor='black', linewidth=2,
                label=f'Default (grounded -> predicts {do})'))
        else:
            legend_handles.append(mpatches.Patch(
                facecolor='#e57373', edgecolor='black', linewidth=2,
                label=f'Default (defeated -> predicts {1-do})'))

    if edges:
        legend_handles.append(mlines.Line2D(
            [], [], color='#cc3333', linewidth=1.5, marker='>',
            markersize=6, label='Attacks'))
    if defense_edges:
        legend_handles.append(mlines.Line2D(
            [], [], color='#2ea02e', linewidth=1.5, marker='>',
            markersize=6, label='New case defends'))

    ax.legend(handles=legend_handles, loc=legend_loc, fontsize=6)
    ax.axis('off')
    # Expand limits so node circles aren't clipped (auto-scale uses centres only)
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    mx, my = (x1 - x0) * 0.12, (y1 - y0) * 0.12
    ax.set_xlim(x0 - mx, x1 + mx)
    ax.set_ylim(y0 - my, y1 + my)


# ── Single-threshold plot ────────────────────────────────────────────────

def plot_single_threshold(models, k, new_case_feat, char_model, strategy='ordinal',
                          figsize=(5, 4.5), save_path=None, show_isolated=False,
                          show_new_case=False, show_defense_arrows=True,
                          exclude_nodes=None, legend_loc='lower left',
                          ranksep=0.9, nodesep=0.4):
    """Display (and optionally save) the argumentation graph for a single threshold k."""
    if strategy == 'flat':
        thresh_label = f'k={k}: outcome=={k}'
    else:
        thresh_label = f'k={k}: outcome>={k}'

    fig, ax = plt.subplots(figsize=figsize)
    plot_argumentation_subgraph(models[k], new_case_feat, char_model, k,
                                ax=ax, title=thresh_label,
                                show_isolated=show_isolated,
                                show_new_case=show_new_case,
                                show_defense_arrows=show_defense_arrows,
                                exclude_nodes=exclude_nodes,
                                legend_loc=legend_loc,
                                ranksep=ranksep, nodesep=nodesep)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f'Saved -> {save_path}')
    plt.show()


# ── Textual explanation (one binary threshold model) ────────────────────

def explain_threshold(model_k, new_case_feat, char_model, k, strategy='ordinal'):
    """Return (text_str, binary_prediction) for threshold k."""
    nf = new_case_feat[np.newaxis, :] if new_case_feat.ndim == 1 else new_case_feat
    attacks_mask = model_k.get_new_case_attacks_mask(nf)[0]
    grounded     = model_k.compute_grounded(attacks_mask[np.newaxis, :])[0]

    default_idx     = model_k.default_index
    binary_outcomes = model_k.casebase_outcomes
    n_non_default   = default_idx

    rel_ids     = list(np.where(~attacks_mask[:n_non_default])[0])
    neutralized = int(attacks_mask[:n_non_default].sum())
    rel_0 = [i for i in rel_ids if binary_outcomes[i] == 0]
    rel_1 = [i for i in rel_ids if binary_outcomes[i] == 1]

    def _fmt(ids, mx=15):
        parts = [f'Case {i} [{_abbrev(model_k.casebase_features[i], char_model, 10)}]'
                 for i in ids[:mx]]
        if len(ids) > mx:
            parts.append(f'+{len(ids)-mx} more')
        return ', '.join(parts) or 'none'

    survived = bool(grounded[default_idx])
    bp       = model_k.default_outcome if survived else 1 - model_k.default_outcome
    status   = 'SURVIVED' if survived else 'DEFEATED'

    if strategy == 'flat':
        q_label = f'outcome == {k}'
    else:
        q_label = f'outcome >= {k}'

    lines = [
        f'  Relevant (features subset of new case): {len(rel_ids)}  |  Neutralised: {neutralized}',
        f'  Support {q_label} (label=1): {_fmt(rel_1)}',
        f'  Oppose  {q_label} (label=0): {_fmt(rel_0)}',
        f'  Default {status} -> binary prediction = {bp}',
    ]
    return '\n'.join(lines), bp


# ── Main: interpret a single new case ───────────────────────────────────

def interpret_new_case(models, n_bins, new_case_feat, char_model,
                       strategy, pid, true_outcome, save_dir=None,
                       k_only=None, show_isolated=False):
    """Show graphs + print explanation for a new test case.

    k_only: if set to an int, only show that threshold's graph.
    show_isolated: if False (default), hide degree-0 nodes.
    Supported strategies: 'ordinal', 'flat'
    """
    if strategy not in ('ordinal', 'flat'):
        print(f'strategy={strategy!r} not supported. Use "ordinal" or "flat".')
        return

    print()
    print('=' * 70)
    print(f'Patient      : {pid}')
    print(f'True outcome : {true_outcome}')
    print(f'Features     : {_abbrev(new_case_feat, char_model, 10)}')
    print(f'Strategy     : {strategy}')
    print('=' * 70)

    k_range = [k_only] if k_only is not None else range(n_bins)

    binary_preds = [None] * n_bins
    fig, axes = plt.subplots(len(k_range), 1,
                             figsize=(16, 7 * len(k_range)), squeeze=False)
    axes_flat = axes.flatten()

    for plot_idx, k in enumerate(k_range):
        if strategy == 'flat':
            thresh_label = f'k={k}: outcome=={k}'
        else:
            thresh_label = f'k={k}: outcome>={k}'
        print()
        print('-' * 50)
        print(f'Model k={k}  ({thresh_label})')
        expl, bp = explain_threshold(models[k], new_case_feat, char_model, k, strategy)
        print(expl)
        binary_preds[k] = bp
        plot_argumentation_subgraph(models[k], new_case_feat, char_model, k,
                                    ax=axes_flat[plot_idx], title=thresh_label,
                                    show_isolated=show_isolated)

    plt.suptitle(f'Argumentation  --  {pid}', fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    if save_dir is not None:
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        suffix = f'_k{k_only}' if k_only is not None else ''
        save_path = Path(save_dir) / f'{pid}_{strategy}{suffix}.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved -> {save_path}')
    plt.show()

    known = [bp for bp in binary_preds if bp is not None]
    if k_only is None:
        print()
        print('=' * 70)
        print('Binary preds: ' + '  '.join(f'k={k}:{bp}'
              for k, bp in enumerate(binary_preds)))
        if strategy == 'ordinal':
            final = sum(binary_preds[1:])
            print(f'Ordinal prediction : {final}  (True: {true_outcome})')
        else:
            fired = [k for k, bp in enumerate(binary_preds) if bp == 1]
            final = max(fired) if fired else n_bins - 1
            print(f'Flat prediction    : {final}  (True: {true_outcome})')
            if not fired:
                print(f'  (no model fired -- defaulting to class {n_bins - 1})')
        print('=' * 70)


print('Helpers loaded.')

---
## BraTS -- OS Survival Prediction

Set `DATA_SOURCE = 'gt'` to use ground-truth segmentation masks (no checkpoint needed),  
or `DATA_SOURCE = 'trained'` to use a trained 2D slot-attention model.

In [13]:
# ── BraTS configuration -- edit this cell ──────────────────────────────────
DATA_SOURCE = 'gt'       # 'gt' | 'trained'
CHECKPOINT  = (
    '/path/to/BrainWear_Kareem/FYP'
    '/slot_attention/training_2d/models/checkpoints/brats_png_v14a_0.15_test/ckpt.pt'
)  # only used when DATA_SOURCE == 'trained'

DATA_DIR = (
    '/path/to/BrainWear_Kareem'
    '/Processed_BraTS2020_TrainingData_PNG'
)
OS_CSV = '/path/to/BrainWear_Kareem/BraTS_OS.csv'

CHAR_MODEL_NAME = 'small'   # 'small' | 'large' | 'small_v2' | 'large_v2'
N_BINS     = 4
AGG_MODE   = 'max'           # 'sum' | 'max'
STRATEGY   = 'ordinal'
STRICT     = False
TRAIN_FRAC = 0.8
SEED       = 0
NUM_SLOTS  = 5

In [5]:
# ── BraTS: load data and build case base ───────────────────────────────────
from datasets.brats2020_png import BraTS2020PNGDataset, batch_seg_to_slot_targets_2d

np.random.seed(SEED)
torch.manual_seed(SEED)
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHAR_MODEL = CHAR_MODELS[CHAR_MODEL_NAME]
cfg        = BraTSOutcomeConfig.from_json(CONFIG)


def _parse_survival_days(raw):
    raw = raw.strip()
    try:
        return float(raw)
    except ValueError:
        m = re.search(r'(\d+)', raw)
        return float(m.group(1)) if m else None


def load_os_csv(csv_path, n_bins=5):
    rows = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv_mod.DictReader(f):
            pid  = row['Brats20ID'].strip()
            days = _parse_survival_days(row['Survival_days'])
            if days is not None:
                rows.append((pid, days))
    if not rows:
        raise ValueError('No usable rows in OS CSV.')
    pids      = [r[0] for r in rows]
    survival  = np.array([r[1] for r in rows])
    quantiles = np.quantile(survival, np.linspace(0, 1, n_bins + 1)[1:-1])
    bins      = np.digitize(survival, quantiles)
    print(f'OS CSV: {len(rows)} patients | {n_bins} bins | thresholds {np.round(quantiles).astype(int).tolist()}')
    print(f'Class distribution: {np.bincount(bins, minlength=n_bins).tolist()}')
    return {p: int(b) for p, b in zip(pids, bins)}, quantiles


def group_by_patient(dataset):
    groups = {}
    for i, (t2_path, _) in enumerate(dataset.samples):
        pid = Path(t2_path).parent.name
        groups.setdefault(pid, []).append(i)
    return groups


def _build_feats_gt(dataset, groups, char_model, num_slots, device, agg_mode):
    feats = {}
    with torch.no_grad():
        for pid, idxs in tqdm(groups.items(), desc=f'GT features ({agg_mode})'):
            acc = char_model.default_case().astype(np.int64)
            for i in idxs:
                _, seg = dataset[i]
                slots = batch_seg_to_slot_targets_2d(seg.unsqueeze(0), num_slots)[0]
                v = char_model.characterisation_transform(slots).astype(np.int64)
                acc = np.maximum(acc, v) if agg_mode == 'max' else acc + v
            feats[pid] = acc
    return feats


def _load_slot_model(ckpt_path, device):
    from slot_attention.training_2d.slot_attention_2d import SlotClassifier2D
    ckpt = torch.load(ckpt_path, map_location=device)
    hp   = ckpt.get('hyperparameters', {})
    model = SlotClassifier2D(
        in_shape=hp.get('in_shape', (1, 240, 240)),
        width=hp.get('width', 64),
        num_slots=hp.get('num_slots', 5),
        slot_dim=hp.get('slot_dim', 64),
        routing_iters=hp.get('routing_iters', 7),
        temperature=hp.get('temperature', 0.5),
        encoder_depth=hp.get('encoder_depth', 4),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(device).eval()
    print(f'Slot model loaded  (epoch {ckpt.get("epoch", "?")})')
    return model


def _build_feats_trained(dataset, groups, char_model, num_slots, device, slot_model, agg_mode):
    feats = {}
    with torch.no_grad():
        for pid, idxs in tqdm(groups.items(), desc=f'Trained features ({agg_mode})'):
            acc = char_model.default_case().astype(np.int64)
            for i in idxs:
                t2, _ = dataset[i]
                _, _, _, _, y_hat = slot_model(t2.unsqueeze(0).to(device))
                slots = y_hat[0].cpu()
                v = char_model.characterisation_transform(slots).astype(np.int64)
                acc = np.maximum(acc, v) if agg_mode == 'max' else acc + v
            feats[pid] = acc
    return feats


# ── load ────────────────────────────────────────────────────────────────────
patient_to_class, _ = load_os_csv(OS_CSV, n_bins=N_BINS)
dataset    = BraTS2020PNGDataset(data_dir=DATA_DIR, is_train=False)
all_groups = group_by_patient(dataset)
matched_pids   = sorted(p for p in all_groups if p in patient_to_class)
matched_groups = {p: all_groups[p] for p in matched_pids}

if DATA_SOURCE == 'gt':
    feat_dict = _build_feats_gt(dataset, matched_groups, CHAR_MODEL, NUM_SLOTS, device, AGG_MODE)
elif DATA_SOURCE == 'trained':
    _slot_model = _load_slot_model(CHECKPOINT, device)
    feat_dict   = _build_feats_trained(dataset, matched_groups, CHAR_MODEL, NUM_SLOTS,
                                        device, _slot_model, AGG_MODE)
else:
    raise ValueError(f'Unknown DATA_SOURCE: {DATA_SOURCE!r}')

all_feats    = np.array([feat_dict[p] for p in matched_pids])
all_outcomes = np.array([patient_to_class[p] for p in matched_pids])

idx = np.arange(len(matched_pids))
if np.any(np.bincount(all_outcomes, minlength=N_BINS) < 2):
    rng  = np.random.default_rng(SEED)
    perm = rng.permutation(len(idx))
    n_tr = int(round(TRAIN_FRAC * len(idx)))
    train_idx, test_idx = perm[:n_tr], perm[n_tr:]
else:
    train_idx, test_idx = train_test_split(
        idx, train_size=TRAIN_FRAC, random_state=SEED, stratify=all_outcomes)

brats_train_feats    = all_feats[train_idx]
brats_train_outcomes = all_outcomes[train_idx]
brats_test_feats     = all_feats[test_idx]
brats_test_outcomes  = all_outcomes[test_idx]
brats_train_pids     = [matched_pids[i] for i in train_idx]
brats_test_pids      = [matched_pids[i] for i in test_idx]

print(f'\nTrain: {len(train_idx)}  |  Test: {len(test_idx)}')
print(f'Unique casebase vectors: {len(np.unique(brats_train_feats, axis=0))} / {len(brats_train_feats)}')

brats_models = _fit_ordinal_models(
    brats_train_feats, brats_train_outcomes,
    CHAR_MODEL, cfg, N_BINS, strict=STRICT, strategy=STRATEGY)
print(f'Fitted {N_BINS} ordinal AACBR models  (DATA_SOURCE={DATA_SOURCE!r}, strict={STRICT})')

In [6]:
# ── BraTS: interpret a new test case ───────────────────────────────────────
# Change NEW_CASE_IDX in the config cell to inspect a different patient.
NEW_CASE_IDX = 5   # index into test set -- change to inspect different patients
new_pid  = brats_test_pids[NEW_CASE_IDX]
new_feat = brats_test_feats[NEW_CASE_IDX]
true_out = brats_test_outcomes[NEW_CASE_IDX]

SAVE_DIR = '/path/to/BrainWear_Kareem/FYP/eval/aacbr_plots'   # set to a directory path string to save PNGs, e.g. '/tmp/brats_plots'

print(f'Test set size: {len(brats_test_pids)}  |  Inspecting index {NEW_CASE_IDX}')
interpret_new_case(
    brats_models, N_BINS, new_feat, CHAR_MODEL,
    STRATEGY, new_pid, true_out, save_dir=SAVE_DIR,
)

In [44]:
plot_single_threshold(brats_models, k=3, new_case_feat=new_feat,
                      char_model=CHAR_MODEL, strategy=STRATEGY,
                      show_new_case=True, show_defense_arrows=False,
                      legend_loc='upper left', exclude_nodes=[3],
                      figsize=(5, 3),
                      save_path=SAVE_DIR + '/k3.png')


---
## BrainWear -- EORTC PRO Score Prediction

Requires a trained 2D slot-attention checkpoint.  
Set `SCORE_NAME` to any of the 26 EORTC score columns (e.g. `'QL2'`, `'PF2'`, `'BNVD'`).

In [ ]:
# ── BrainWear configuration -- edit this cell ──────────────────────────────
BW_CHECKPOINT = (
    '/path/to/BrainWear_Kareem/FYP'
    '/slot_attention/training_2d/models/checkpoints/brats_png_v14a_gamma4/ckpt.pt'
)
BW_DATA_DIR = (
    '/path/to/BrainWear_Kareem'
    '/Processed_Brainwear_PNG'
)
BW_SCORE_FILE = (
    '/path/to/BrainWear_Kareem'
    '/eortc_scores.csv'
)

SCORE_NAME    = 'QL2'    # any EORTC score column
BW_CHAR_NAME  = 'small'  # 'small' | 'large' | 'small_v2' | 'large_v2'
BW_N_BINS     = 3
BW_AGG_MODE   = 'sum'    # 'sum' | 'max'
BW_STRATEGY   = 'ordinal'
BW_STRICT     = True
BW_TRAIN_FRAC = 0.8
BW_SEED       = 0
BW_NUM_SLOTS  = 5

BW_NEW_CASE_IDX = 0

In [ ]:
# ── BrainWear: load data and build case base ───────────────────────────────
from PIL import Image
import torchvision.transforms.functional as TF

np.random.seed(BW_SEED)
torch.manual_seed(BW_SEED)
bw_device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BW_CHAR_MODEL = CHAR_MODELS[BW_CHAR_NAME]
bw_cfg        = BraTSOutcomeConfig.from_json(CONFIG)


def _load_eortc_scores(score_file):
    scores = {}
    with open(score_file, newline='', encoding='utf-8') as f:
        reader = csv_mod.DictReader(f)
        q_col  = reader.fieldnames[0]
        pids   = [h.strip() for h in reader.fieldnames[1:] if h.strip()]
        for row in reader:
            name = row[q_col].strip()
            if not name or name == 'Date':
                continue
            vals = {}
            for pid in pids:
                try:
                    vals[pid] = float(row[pid].strip())
                except (ValueError, KeyError, TypeError):
                    pass
            if vals:
                scores[name] = vals
    print(f'Loaded {len(scores)} EORTC scores for {len(pids)} patients.')
    return scores


def _get_bw_outcomes(eortc_scores, score_name, n_bins, available_pids):
    sd    = eortc_scores.get(score_name, {})
    valid = [p for p in available_pids if p in sd]
    if not valid:
        raise ValueError(f'No patients found for score {score_name!r}.')
    raw       = np.array([sd[p] for p in valid])
    quantiles = np.quantile(raw, np.linspace(0, 1, n_bins + 1)[1:-1])
    bins      = np.digitize(raw, quantiles)
    print(f'{score_name}: {len(valid)} patients | thresholds {np.round(quantiles, 1).tolist()}')
    print(f'Class distribution: {np.bincount(bins, minlength=n_bins).tolist()}')
    return {p: int(b) for p, b in zip(valid, bins)}


def _find_axial_t2(patient_dir):
    candidates = sorted(d for d in patient_dir.iterdir()
                        if d.is_dir() and d.name.startswith('Axial_T2'))
    return candidates[0] if candidates else None


def _extract_slot_outputs_bw(data_dir, slot_model, device):
    cache = {}
    patient_dirs = sorted(p for p in Path(data_dir).iterdir() if p.is_dir())
    with torch.no_grad():
        for pd_ in tqdm(patient_dirs, desc='Slot outputs (BrainWear)'):
            pid       = pd_.name
            axial_dir = _find_axial_t2(pd_)
            if axial_dir is None:
                continue
            slices = sorted(axial_dir.glob('*.png'))
            if not slices:
                continue
            pid_out = []
            for sl in slices:
                t2 = TF.to_tensor(Image.open(sl).convert('L'))
                _, _, _, _, y_hat = slot_model(t2.unsqueeze(0).to(device))
                pid_out.append(y_hat[0].cpu())
            cache[pid] = pid_out
    print(f'Slot outputs: {len(cache)} patients, ' +
          f'{sum(len(v) for v in cache.values())} slices.')
    return cache


def _compute_char_feats_bw(slot_cache, char_model, agg_mode):
    features = {}
    for pid, slices in slot_cache.items():
        acc = char_model.default_case().astype(np.int64)
        for slots in slices:
            v = char_model.characterisation_transform(slots).astype(np.int64)
            acc = np.maximum(acc, v) if agg_mode == 'max' else acc + v
        features[pid] = acc
    return features


# ── load ────────────────────────────────────────────────────────────────────
bw_slot_model = _load_slot_model(BW_CHECKPOINT, bw_device)
eortc_scores  = _load_eortc_scores(BW_SCORE_FILE)
bw_slot_cache = _extract_slot_outputs_bw(BW_DATA_DIR, bw_slot_model, bw_device)
bw_all_pids   = list(bw_slot_cache.keys())

pid_to_class  = _get_bw_outcomes(eortc_scores, SCORE_NAME, BW_N_BINS, bw_all_pids)
bw_feat_dict  = _compute_char_feats_bw(bw_slot_cache, BW_CHAR_MODEL, BW_AGG_MODE)

bw_matched  = [p for p in bw_feat_dict if p in pid_to_class]
bw_feats    = np.array([bw_feat_dict[p] for p in bw_matched])
bw_outcomes = np.array([pid_to_class[p] for p in bw_matched])

bw_idx = np.arange(len(bw_matched))
if np.any(np.bincount(bw_outcomes, minlength=BW_N_BINS) < 2):
    rng  = np.random.default_rng(BW_SEED)
    perm = rng.permutation(len(bw_idx))
    n_tr = int(round(BW_TRAIN_FRAC * len(bw_idx)))
    bw_train_idx, bw_test_idx = perm[:n_tr], perm[n_tr:]
else:
    bw_train_idx, bw_test_idx = train_test_split(
        bw_idx, train_size=BW_TRAIN_FRAC, random_state=BW_SEED, stratify=bw_outcomes)

bw_train_feats    = bw_feats[bw_train_idx]
bw_train_outcomes = bw_outcomes[bw_train_idx]
bw_test_feats     = bw_feats[bw_test_idx]
bw_test_outcomes  = bw_outcomes[bw_test_idx]
bw_train_pids     = [bw_matched[i] for i in bw_train_idx]
bw_test_pids      = [bw_matched[i] for i in bw_test_idx]

print(f'\nTrain: {len(bw_train_idx)}  |  Test: {len(bw_test_idx)}')
print(f'Unique casebase vectors: {len(np.unique(bw_train_feats, axis=0))} / {len(bw_train_feats)}')

bw_models = _fit_ordinal_models(
    bw_train_feats, bw_train_outcomes,
    BW_CHAR_MODEL, bw_cfg, BW_N_BINS, strict=BW_STRICT, strategy=BW_STRATEGY)
print(f'Fitted {BW_N_BINS} ordinal AACBR models  (score={SCORE_NAME!r}, strict={BW_STRICT})')

In [ ]:
# ── BrainWear: interpret a new test case ───────────────────────────────────
# Change BW_NEW_CASE_IDX in the config cell to inspect a different patient.
bw_pid  = bw_test_pids[BW_NEW_CASE_IDX]
bw_feat = bw_test_feats[BW_NEW_CASE_IDX]
bw_true = bw_test_outcomes[BW_NEW_CASE_IDX]

BW_SAVE_DIR = None   # set to a directory path string to save PNGs

print(f'Test set size: {len(bw_test_pids)}  |  Inspecting index {BW_NEW_CASE_IDX}')
interpret_new_case(
    bw_models, BW_N_BINS, bw_feat, BW_CHAR_MODEL,
    BW_STRATEGY, bw_pid, bw_true, save_dir=BW_SAVE_DIR,
)